# Latent shape math and one DDIM step

In [1]:
import numpy as np

def latent_shape(H, W, C=3, factor=8, latent_channels=4):
    return (latent_channels, H // factor, W // factor)

def pixel_shape(lat_h, lat_w, factor=8, out_channels=3):
    return (out_channels, lat_h * factor, lat_w * factor)

def compression_ratio(H, W, C=3, factor=8, latent_channels=4):
    return float((H * W * C) / (latent_channels * (H // factor) * (W // factor)))

def predict_x0(x_t, eps, alpha_t):
    return (x_t - np.sqrt(1.0 - alpha_t) * eps) / np.sqrt(alpha_t)

def ddim_step(x_t, eps, alpha_t, alpha_prev):
    x0 = predict_x0(x_t, eps, alpha_t)
    return np.sqrt(alpha_prev) * x0 + np.sqrt(1.0 - alpha_prev) * eps

In [2]:
print(latent_shape(512, 512))        # (4, 64, 64)
print(latent_shape(256, 384))        # (4, 32, 48)
print(pixel_shape(64, 64))           # (3, 512, 512)
print(compression_ratio(512, 512))   # 48.0

a = 0.6
x0 = np.random.RandomState(1).randn(4, 8, 8)
eps = np.random.RandomState(2).randn(4, 8, 8)
x_t = np.sqrt(a) * x0 + np.sqrt(1 - a) * eps
print(np.allclose(predict_x0(x_t, eps, a), x0))      # True
print(np.allclose(ddim_step(x_t, eps, a, a), x_t))   # True

x_t = np.random.RandomState(3).randn(4, 8, 8)
print(np.allclose(ddim_step(x_t, np.zeros_like(x_t), 1.0, 0.4), np.sqrt(0.4) * x_t))  # True
print(ddim_step(x_t, eps, 0.7, 0.4).shape)           # (4, 8, 8)

(4, 64, 64)
(4, 32, 48)
(3, 512, 512)
48.0
True
True
True
(4, 8, 8)
